In [1]:
# %%
"""
IA3 fine-tuning for MedGemma-4B on pooled RAW EEG topomaps, with blinded
patient-wise evaluation.

Notebook usage:
1. Put this whole file into a Jupyter notebook cell, or open it as a
   cell-marked Python notebook in VS Code/Jupyter.
2. Confirm the CSV paths below.
3. Run top to bottom.

Labels:
    0 = non-seizure
    1 = seizure
"""

# %%
# =========================
# 0. ENVIRONMENT SETTINGS
# =========================

import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from pathlib import Path
from datetime import datetime
import gc
import json
import math
import random
import re
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from transformers import AutoProcessor, get_linear_schedule_with_warmup

try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    # Fallback for older Transformers builds.
    from transformers import AutoModelForVision2Seq as AutoModelForImageTextToText

from peft import IA3Config, PeftModel, TaskType, get_peft_model


# %%
# =========================
# 1. USER CONFIGURATION
# =========================

MODEL_ID = "google/medgemma-4b-it"

TRAIN_CSV = r"C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\Raw\All training together RAw\training_metadata.csv"

PATIENT_TEST_CSVS = {
    "08": r"C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\Patient 08\RAW\Testing\bLINDED\Patient_8_RAW_blinded_test_labels.csv",
    # You gave this exact path. The filename says Patient_18, so the code warns you if that is accidental.
    "11": r"C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\Patient 11\Raw\Testing\Blinded\Patient_18_blinded_test_labels.csv",
    "18": r"C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\Patient 18\RAW\Testing\Blinded\Patient_18_RAW_blinded_test_labels.csv",
    "24": r"C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\Patiient 24\Raw Fine Tuning\Testing\Patient_24_RAW_blinded_test_labels.csv",
}

CLASSIFICATION_PROMPT = """You are an expert in identifying seizure patterns from EEG variance topomaps.
The attached image represents a 2 second EEG segment
Classify the provided fixed-scale EEG variance topomap as one of the following:
seizure
non-seizure
The raw EEG signal was divided into fixed time windows. For each window, the variance of every bipolar EEG channel was calculated directly from the raw EEG values. Each channel variance value was positioned at the midpoint between the two electrodes forming that bipolar channel. These values were spatially interpolated to create the EEG topomap.
All topomaps use the same fixed color scale, allowing direct comparison across images. Yellow regions indicate higher EEG variance, while dark purple regions indicate lower EEG variance.
Classify the image based on the visible variance intensity and spatial distribution. Consider whether high-variance activity is focal or widespread and symmetric or asymmetric. Evaluate these characteristics together rather than using any single feature as a definitive rule.

Answer with exactly one label: seizure or non-seizure."""

LABEL_ID_TO_TEXT = {
    0: "non-seizure",
    1: "seizure",
}

# Training hyperparameters.
SEED = 42
VALIDATION_SIZE = 0.20
EPOCHS = 50
PATIENCE = 7
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 8e-4
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
USE_GRADIENT_CHECKPOINTING = False

# IA3 placement.
# This script targets language-side attention k/v projections and FFN down_proj.
# It intentionally excludes vision modules so RAW image interpretation stays stable.
IA3_SUFFIXES = ("k_proj", "v_proj", "down_proj")
IA3_FEEDFORWARD_SUFFIXES = ("down_proj",)
EXCLUDE_MODULE_KEYWORDS = ("vision", "visual", "image", "projector", "multi_modal")
LANGUAGE_MODULE_KEYWORDS = ("language_model", "text_model", "model.layers")

# Evaluation.
# "generate" is closest to your normal prompt-based classification.
# "likelihood" is deterministic and often faster for validation/debugging.
VAL_EVAL_MODE = "generate"
TEST_EVAL_MODE = "generate"
MAX_NEW_TOKENS = 8

# Jupyter on Windows is safest with NUM_WORKERS = 0.
NUM_WORKERS = 0
PIN_MEMORY = True

# Use DEBUG_MAX_* only for a smoke test. Keep as None for the real experiment.
DEBUG_MAX_TRAIN_ROWS = None
DEBUG_MAX_VAL_ROWS = None
DEBUG_MAX_TEST_ROWS = None

VALIDATE_IMAGE_PATHS_BEFORE_TRAINING = True
SAVE_EPOCH_VALIDATION_PREDICTIONS = False

# Optional path remapping if CSVs contain old image roots.
# Example:
# IMAGE_ROOT_REMAPS = {
#     r"C:\old\folder": r"D:\new\folder",
# }
IMAGE_ROOT_REMAPS: Dict[str, str] = {}


# %%
# =========================
# 2. REPRODUCIBILITY / DEVICE
# =========================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This MedGemma IA3 script expects an NVIDIA GPU.")

DEVICE = torch.device("cuda")
DTYPE = torch.bfloat16

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("GPU:", torch.cuda.get_device_name(0))
print("Precision:", DTYPE)


# %%
# =========================
# 3. OUTPUT FOLDER
# =========================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_ROOT = Path(TRAIN_CSV).parent / "MedGemma4B_IA3_Pooled_RAW"
RUN_DIR = OUTPUT_ROOT / f"run_{timestamp}"
BEST_ADAPTER_DIR = RUN_DIR / "best_ia3_adapter"
EVAL_DIR = RUN_DIR / "patient_evaluations"

RUN_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Run folder:")
print(RUN_DIR)


# %%
# =========================
# 4. CSV AND IMAGE HELPERS
# =========================

IMAGE_COLUMN_CANDIDATES = [
    "image_path",
    "blinded_image_path",
    "topomap_path",
    "file_path",
    "filepath",
    "path",
    "png_path",
    "jpg_path",
    "image",
    "filename",
    "file_name",
    "blinded_filename",
    "blinded_file_name",
]

LABEL_COLUMN_CANDIDATES = [
    "label",
    "labels",
    "class",
    "target",
    "y",
    "true_label",
    "ground_truth",
    "seizure_label",
    "is_seizure",
    "binary_label",
]

_filename_search_cache: Dict[str, Dict[str, str]] = {}


def find_column(df: pd.DataFrame, candidates: List[str], required: bool, kind: str) -> Optional[str]:
    lower_to_real = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_to_real:
            return lower_to_real[cand.lower()]

    # Fuzzy fallback.
    for c in df.columns:
        c_low = str(c).strip().lower()
        if kind == "image" and ("image" in c_low or "filename" in c_low or c_low.endswith("path")):
            return c
        if kind == "label" and ("label" in c_low or "class" in c_low or "target" in c_low):
            return c

    if required:
        raise ValueError(
            f"Could not find a {kind} column. Available columns:\n{list(df.columns)}"
        )
    return None


def label_to_id(value) -> int:
    if pd.isna(value):
        raise ValueError("Missing label")

    if isinstance(value, (int, np.integer)):
        if int(value) in (0, 1):
            return int(value)
    if isinstance(value, (float, np.floating)) and not math.isnan(float(value)):
        if int(value) in (0, 1) and float(value) == int(value):
            return int(value)

    text = str(value).strip().lower()
    text = text.replace("_", "-")
    text_compact = re.sub(r"[\s\-]+", "", text)

    if text in {"0", "0.0", "false", "normal", "non seizure", "non-seizure", "nonseizure"}:
        return 0
    if text in {"1", "1.0", "true", "ictal", "seizure"}:
        return 1

    # Check non-seizure before seizure because "non-seizure" contains "seizure".
    if "non" in text_compact and "seiz" in text_compact:
        return 0
    if "seiz" in text_compact or "ictal" in text_compact:
        return 1

    raise ValueError(f"Cannot convert label value to 0/1: {value!r}")


def apply_root_remap(raw_path: str) -> str:
    normalized_raw = raw_path.replace("/", "\\")
    for old_root, new_root in IMAGE_ROOT_REMAPS.items():
        old_norm = old_root.replace("/", "\\").rstrip("\\")
        new_norm = new_root.rstrip("\\/")
        if normalized_raw.lower().startswith(old_norm.lower()):
            suffix = normalized_raw[len(old_norm):].lstrip("\\/")
            return str(Path(new_norm) / suffix)
    return raw_path


def build_filename_index(root: Path) -> Dict[str, str]:
    key = str(root.resolve()) if root.exists() else str(root)
    if key in _filename_search_cache:
        return _filename_search_cache[key]

    index: Dict[str, str] = {}
    if root.exists():
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.webp"):
            for p in root.rglob(ext):
                index.setdefault(p.name.lower(), str(p))

    _filename_search_cache[key] = index
    return index


def resolve_image_path(value, csv_path: Path) -> str:
    if pd.isna(value):
        return ""

    raw = str(value).strip().strip('"').strip("'")
    raw = apply_root_remap(raw)
    p = Path(raw)

    if p.exists():
        return str(p)

    csv_parent = csv_path.parent
    candidate = csv_parent / raw
    if candidate.exists():
        return str(candidate)

    # If the CSV gives only a filename, search near the CSV.
    filename = Path(raw).name
    if filename:
        index = build_filename_index(csv_parent)
        found = index.get(filename.lower())
        if found:
            return found

    return raw


def validate_existing_images(df: pd.DataFrame, dataset_name: str) -> None:
    missing = []
    for p in tqdm(df["image_path"].tolist(), desc=f"Validate images: {dataset_name}"):
        if not Path(p).exists():
            missing.append(p)

    if missing:
        example = "\n".join(missing[:10])
        raise FileNotFoundError(
            f"{dataset_name}: {len(missing)} image files are missing. First examples:\n{example}\n\n"
            "Fix the image_path/blinded_image_path column or add IMAGE_ROOT_REMAPS near the top."
        )


def load_metadata(csv_file: str, dataset_name: str, debug_max_rows: Optional[int] = None) -> pd.DataFrame:
    csv_path = Path(csv_file)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"{dataset_name} CSV not found:\n{csv_path}\n\n"
            "Check spelling, folder capitalization, and OneDrive sync status."
        )

    df_raw = pd.read_csv(csv_path)
    image_col = find_column(df_raw, IMAGE_COLUMN_CANDIDATES, required=True, kind="image")
    label_col = find_column(df_raw, LABEL_COLUMN_CANDIDATES, required=True, kind="label")

    df = pd.DataFrame()
    df["source_csv"] = str(csv_path)
    df["source_row"] = np.arange(len(df_raw))
    df["image_path"] = df_raw[image_col].apply(lambda x: resolve_image_path(x, csv_path))
    df["label"] = df_raw[label_col].apply(label_to_id)

    # Keep useful identity columns if present.
    for optional_col in [
        "blinded_id",
        "blinded_filename",
        "filename",
        "file_name",
        "Global_Serial",
        "global_serial",
        "window_start",
        "window_end",
    ]:
        if optional_col in df_raw.columns and optional_col not in df.columns:
            df[optional_col] = df_raw[optional_col]

    df = df.dropna(subset=["image_path", "label"]).copy()
    df["label"] = df["label"].astype(int)
    df["label_text"] = df["label"].map(LABEL_ID_TO_TEXT)

    if debug_max_rows is not None:
        df = df.sample(min(debug_max_rows, len(df)), random_state=SEED).reset_index(drop=True)

    print(f"\n{dataset_name}: {len(df)} rows")
    print("CSV:", csv_path)
    print("Image column:", image_col)
    print("Label column:", label_col)
    print("Class counts:", df["label"].value_counts().sort_index().to_dict())

    return df.reset_index(drop=True)


# %%
# =========================
# 5. LOAD DATA
# =========================

train_all_df = load_metadata(TRAIN_CSV, "pooled RAW training", DEBUG_MAX_TRAIN_ROWS)

if VALIDATE_IMAGE_PATHS_BEFORE_TRAINING:
    validate_existing_images(train_all_df, "pooled RAW training")

train_df, val_df = train_test_split(
    train_all_df,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=train_all_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

if DEBUG_MAX_VAL_ROWS is not None:
    val_df = val_df.sample(min(DEBUG_MAX_VAL_ROWS, len(val_df)), random_state=SEED).reset_index(drop=True)

print("\nTrain split:", len(train_df), train_df["label"].value_counts().sort_index().to_dict())
print("Val split:", len(val_df), val_df["label"].value_counts().sort_index().to_dict())


# %%
# =========================
# 6. PROCESSOR AND PROMPT BUILDERS
# =========================

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

if getattr(processor, "tokenizer", None) is not None:
    tokenizer = processor.tokenizer
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
else:
    tokenizer = None


def decode_tokens(token_ids) -> str:
    if tokenizer is not None:
        return tokenizer.decode(token_ids, skip_special_tokens=True)
    return processor.decode(token_ids, skip_special_tokens=True)


def make_prompt_messages(image: Image.Image) -> List[dict]:
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": CLASSIFICATION_PROMPT},
            ],
        }
    ]


def make_supervised_messages(image: Image.Image, answer_text: str) -> List[dict]:
    messages = make_prompt_messages(image)
    messages.append(
        {
            "role": "assistant",
            "content": [{"type": "text", "text": answer_text}],
        }
    )
    return messages


def move_batch_to_device(batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    moved = {}
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            if torch.is_floating_point(v):
                moved[k] = v.to(DEVICE, dtype=DTYPE, non_blocking=True)
            else:
                moved[k] = v.to(DEVICE, non_blocking=True)
        else:
            moved[k] = v
    return moved


def build_prompt_inputs(image: Image.Image) -> Dict[str, torch.Tensor]:
    inputs = processor.apply_chat_template(
        make_prompt_messages(image),
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    return dict(inputs)


def build_supervised_inputs(image: Image.Image, answer_text: str) -> Dict[str, torch.Tensor]:
    full_inputs = processor.apply_chat_template(
        make_supervised_messages(image, answer_text),
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    prompt_inputs = processor.apply_chat_template(
        make_prompt_messages(image),
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    full_inputs = dict(full_inputs)
    labels = full_inputs["input_ids"].clone()
    prompt_len = prompt_inputs["input_ids"].shape[-1]
    labels[:, :prompt_len] = -100

    if "attention_mask" in full_inputs:
        labels[full_inputs["attention_mask"] == 0] = -100

    full_inputs["labels"] = labels
    return full_inputs


# %%
# =========================
# 7. DATASET / DATALOADER
# =========================

class TopomapSupervisedDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        image_path = row["image_path"]
        image = Image.open(image_path).convert("RGB")
        answer_text = LABEL_ID_TO_TEXT[int(row["label"])]
        inputs = build_supervised_inputs(image, answer_text)
        return inputs


def collate_single_item(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    if len(batch) != 1:
        raise ValueError("This notebook is configured for TRAIN_BATCH_SIZE = 1.")
    return batch[0]


train_loader = DataLoader(
    TopomapSupervisedDataset(train_df),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_single_item,
)


# %%
# =========================
# 8. MODEL + IA3 ADAPTER
# =========================

def load_base_model():
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
        device_map=None,
        low_cpu_mem_usage=False,
        trust_remote_code=True,
        attn_implementation="eager",
    )
    model.to(DEVICE)
    model.config.use_cache = False
    return model


def find_ia3_target_modules(model) -> Tuple[List[str], List[str]]:
    target_modules = []
    feedforward_modules = []

    for module_name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue

        lower_name = module_name.lower()
        if any(excluded in lower_name for excluded in EXCLUDE_MODULE_KEYWORDS):
            continue
        if not any(key in lower_name for key in LANGUAGE_MODULE_KEYWORDS):
            continue
        if not module_name.endswith(IA3_SUFFIXES):
            continue

        target_modules.append(module_name)
        if module_name.endswith(IA3_FEEDFORWARD_SUFFIXES):
            feedforward_modules.append(module_name)

    if not target_modules:
        raise RuntimeError(
            "No IA3 target modules were found. Print model.named_modules() and update "
            "LANGUAGE_MODULE_KEYWORDS / IA3_SUFFIXES for this Transformers version."
        )
    if not feedforward_modules:
        raise RuntimeError("No IA3 feedforward modules were found.")

    return target_modules, feedforward_modules


base_model = load_base_model()
target_modules, feedforward_modules = find_ia3_target_modules(base_model)

print(f"IA3 target modules: {len(target_modules)}")
print(f"IA3 feedforward modules: {len(feedforward_modules)}")
print("First few IA3 modules:")
for name in target_modules[:12]:
    print(" ", name)

ia3_config = IA3Config(
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
    feedforward_modules=feedforward_modules,
)

model = get_peft_model(base_model, ia3_config)
model.to(DEVICE)
model.print_trainable_parameters()

if USE_GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if USE_GRADIENT_CHECKPOINTING and hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

meta_params = [name for name, param in model.named_parameters() if param.device.type == "meta"]
if meta_params:
    raise RuntimeError(
        "Some model parameters are still on the meta device. First examples:\n"
        + "\n".join(meta_params[:20])
        + "\n\nRestart the kernel and rerun from the beginning. If this repeats, set "
        "device_map=None and low_cpu_mem_usage=False in from_pretrained."
    )


# %%
# =========================
# 9. OPTIMIZER / SCHEDULER
# =========================

trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_training_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print("Steps per epoch:", steps_per_epoch)
print("Total optimizer steps:", total_training_steps)
print("Warmup steps:", warmup_steps)


# %%
# =========================
# 10. PREDICTION / METRICS
# =========================

def parse_prediction_text(text: str) -> Optional[int]:
    t = text.strip().lower()
    t = t.replace("_", "-")
    compact = re.sub(r"[\s\-]+", "", t)

    if "non" in compact and "seiz" in compact:
        return 0
    if re.search(r"\bseizure\b", t) or "seizure" in compact:
        return 1
    return None


@torch.inference_mode()
def predict_one_generate(model, image: Image.Image) -> Tuple[int, str, dict]:
    model.eval()
    inputs = build_prompt_inputs(image)
    input_len = inputs["input_ids"].shape[-1]
    inputs = move_batch_to_device(inputs)

    with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=True):
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id if tokenizer is not None else None,
            eos_token_id=tokenizer.eos_token_id if tokenizer is not None else None,
        )

    new_tokens = generated[0, input_len:].detach().cpu()
    raw_text = decode_tokens(new_tokens).strip()
    pred = parse_prediction_text(raw_text)

    if pred is not None:
        return pred, raw_text, {}

    # Fallback if generation gives an unexpected phrase.
    fallback_pred, scores = predict_one_likelihood(model, image)
    return fallback_pred, raw_text, scores


@torch.inference_mode()
def score_candidate_answer(model, image: Image.Image, answer_text: str) -> float:
    model.eval()
    inputs = build_supervised_inputs(image, answer_text)
    inputs = move_batch_to_device(inputs)

    with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=True):
        outputs = model(**inputs)

    return float(outputs.loss.detach().cpu())


@torch.inference_mode()
def predict_one_likelihood(model, image: Image.Image) -> Tuple[int, dict]:
    scores = {
        0: score_candidate_answer(model, image, LABEL_ID_TO_TEXT[0]),
        1: score_candidate_answer(model, image, LABEL_ID_TO_TEXT[1]),
    }
    pred = min(scores, key=scores.get)
    return int(pred), {f"nll_{LABEL_ID_TO_TEXT[k]}": v for k, v in scores.items()}


def compute_metrics(y_true: List[int], y_pred: List[int]) -> dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "sensitivity_recall_seizure": sensitivity,
        "specificity_recall_nonseizure": specificity,
        "precision_seizure": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_seizure": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def evaluate_dataframe(
    model,
    df: pd.DataFrame,
    dataset_name: str,
    mode: str,
    save_predictions_path: Optional[Path] = None,
) -> Tuple[dict, pd.DataFrame]:
    y_true: List[int] = []
    y_pred: List[int] = []
    rows: List[dict] = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluate {dataset_name}"):
        image = Image.open(row["image_path"]).convert("RGB")

        if mode == "generate":
            pred, raw_text, extra = predict_one_generate(model, image)
        elif mode == "likelihood":
            pred, extra = predict_one_likelihood(model, image)
            raw_text = LABEL_ID_TO_TEXT[pred]
        else:
            raise ValueError("mode must be 'generate' or 'likelihood'")

        true_label = int(row["label"])
        y_true.append(true_label)
        y_pred.append(pred)

        out = {
            "dataset": dataset_name,
            "source_csv": row.get("source_csv", ""),
            "source_row": int(row.get("source_row", -1)),
            "image_path": row["image_path"],
            "true_label": true_label,
            "true_label_text": LABEL_ID_TO_TEXT[true_label],
            "pred_label": pred,
            "pred_label_text": LABEL_ID_TO_TEXT[pred],
            "raw_generation": raw_text,
        }

        for c in ["blinded_id", "blinded_filename", "filename", "file_name", "Global_Serial", "global_serial"]:
            if c in row:
                out[c] = row[c]

        out.update(extra)
        rows.append(out)

    metrics = compute_metrics(y_true, y_pred)
    pred_df = pd.DataFrame(rows)

    if save_predictions_path is not None:
        pred_df.to_csv(save_predictions_path, index=False)

    return metrics, pred_df


def print_metrics(title: str, metrics: dict) -> None:
    print(f"\n{title}")
    print(f"Accuracy:     {metrics['accuracy'] * 100:.2f}%")
    print(f"Balanced Acc: {metrics['balanced_accuracy'] * 100:.2f}%")
    print(f"Sensitivity:  {metrics['sensitivity_recall_seizure'] * 100:.2f}%")
    print(f"Specificity:  {metrics['specificity_recall_nonseizure'] * 100:.2f}%")
    print(f"Precision:    {metrics['precision_seizure'] * 100:.2f}%")
    print(f"F1:           {metrics['f1_seizure']:.4f}")
    print(f"CM [[TN, FP], [FN, TP]] = [[{metrics['tn']}, {metrics['fp']}], [{metrics['fn']}, {metrics['tp']}]]")


# %%
# =========================
# 11. SAVE RUN CONFIG
# =========================

run_config = {
    "model_id": MODEL_ID,
    "train_csv": TRAIN_CSV,
    "patient_test_csvs": PATIENT_TEST_CSVS,
    "prompt": CLASSIFICATION_PROMPT,
    "seed": SEED,
    "validation_size": VALIDATION_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "max_grad_norm": MAX_GRAD_NORM,
    "dtype": str(DTYPE),
    "val_eval_mode": VAL_EVAL_MODE,
    "test_eval_mode": TEST_EVAL_MODE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "ia3_target_module_count": len(target_modules),
    "ia3_feedforward_module_count": len(feedforward_modules),
    "ia3_target_modules": target_modules,
    "ia3_feedforward_modules": feedforward_modules,
}

with open(RUN_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)


# %%
# =========================
# 12. TRAINING LOOP
# =========================

best_val_ba = -1.0
best_val_loss = float("inf")
epochs_without_improvement = 0
history = []
global_step = 0

print("\nStarting IA3 training...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    epoch_loss_sum = 0.0
    epoch_examples = 0
    start_time = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for step, batch in enumerate(pbar, start=1):
        batch = move_batch_to_device(batch)

        with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=True):
            outputs = model(**batch)
            loss = outputs.loss
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaled_loss.backward()

        epoch_loss_sum += float(loss.detach().cpu())
        epoch_examples += 1

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        current_lr = scheduler.get_last_lr()[0]
        pbar.set_postfix(
            {
                "loss": f"{epoch_loss_sum / max(epoch_examples, 1):.4f}",
                "lr": f"{current_lr:.2e}",
            }
        )

    train_loss = epoch_loss_sum / max(epoch_examples, 1)

    val_pred_path = None
    if SAVE_EPOCH_VALIDATION_PREDICTIONS:
        val_pred_path = RUN_DIR / f"val_predictions_epoch_{epoch:03d}.csv"

    val_metrics, _ = evaluate_dataframe(
        model,
        val_df,
        dataset_name=f"validation_epoch_{epoch}",
        mode=VAL_EVAL_MODE,
        save_predictions_path=val_pred_path,
    )

    val_ba = val_metrics["balanced_accuracy"]
    elapsed_min = (time.time() - start_time) / 60.0

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_ba,
        "val_sensitivity": val_metrics["sensitivity_recall_seizure"],
        "val_specificity": val_metrics["specificity_recall_nonseizure"],
        "val_f1": val_metrics["f1_seizure"],
        "val_tn": val_metrics["tn"],
        "val_fp": val_metrics["fp"],
        "val_fn": val_metrics["fn"],
        "val_tp": val_metrics["tp"],
        "lr": scheduler.get_last_lr()[0],
        "elapsed_min": elapsed_min,
    }
    history.append(record)
    pd.DataFrame(history).to_csv(RUN_DIR / "training_history.csv", index=False)

    print_metrics(f"Epoch {epoch} validation", val_metrics)
    print(f"Train loss: {train_loss:.4f} | Epoch time: {elapsed_min:.2f} min")

    improved = (val_ba > best_val_ba) or (
        math.isclose(val_ba, best_val_ba, rel_tol=0.0, abs_tol=1e-12)
        and train_loss < best_val_loss
    )

    if improved:
        best_val_ba = val_ba
        best_val_loss = train_loss
        epochs_without_improvement = 0
        BEST_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(BEST_ADAPTER_DIR)
        processor.save_pretrained(BEST_ADAPTER_DIR)
        print(f"Saved best IA3 adapter to: {BEST_ADAPTER_DIR}")
    else:
        epochs_without_improvement += 1
        print(f"No improvement. Patience: {epochs_without_improvement}/{PATIENCE}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best validation BA = {best_val_ba * 100:.2f}%")
        break

    torch.cuda.empty_cache()

print("\nTraining finished.")
print(f"Best validation balanced accuracy: {best_val_ba * 100:.2f}%")
print(f"Best adapter folder: {BEST_ADAPTER_DIR}")


# %%
# =========================
# 13. RELOAD BEST ADAPTER
# =========================

del model
del base_model
gc.collect()
torch.cuda.empty_cache()

base_model = load_base_model()
model = PeftModel.from_pretrained(base_model, BEST_ADAPTER_DIR)
model.to(DEVICE)
model.eval()

print("Reloaded best IA3 adapter.")


# %%
# =========================
# 14. PATIENT-WISE BLINDED EVALUATION
# =========================

all_patient_metrics = []

for patient_id, csv_path in PATIENT_TEST_CSVS.items():
    csv_name = Path(csv_path).name
    if patient_id not in csv_name.replace("_8_", "_08_"):
        print(
            f"\nWARNING: patient key is {patient_id}, but CSV filename is:\n{csv_name}\n"
            "If this is intentional, ignore this warning. Otherwise fix PATIENT_TEST_CSVS."
        )

    test_df = load_metadata(
        csv_path,
        dataset_name=f"Patient {patient_id} RAW blinded test",
        debug_max_rows=DEBUG_MAX_TEST_ROWS,
    )

    if VALIDATE_IMAGE_PATHS_BEFORE_TRAINING:
        validate_existing_images(test_df, f"Patient {patient_id} RAW blinded test")

    pred_path = EVAL_DIR / f"patient_{patient_id}_RAW_IA3_predictions.csv"
    metrics, pred_df = evaluate_dataframe(
        model,
        test_df,
        dataset_name=f"patient_{patient_id}",
        mode=TEST_EVAL_MODE,
        save_predictions_path=pred_path,
    )

    print_metrics(f"Patient {patient_id} RAW IA3", metrics)

    row = {"patient": patient_id, "csv": csv_path, "predictions_csv": str(pred_path)}
    row.update(metrics)
    all_patient_metrics.append(row)

    pd.DataFrame(all_patient_metrics).to_csv(EVAL_DIR / "summary_metrics_so_far.csv", index=False)
    torch.cuda.empty_cache()

summary_df = pd.DataFrame(all_patient_metrics)
summary_path = EVAL_DIR / "summary_metrics.csv"
summary_df.to_csv(summary_path, index=False)

with open(EVAL_DIR / "summary_metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_patient_metrics, f, indent=2)

print("\nFinal patient-wise summary:")
display(summary_df)
print("Saved summary to:")
print(summary_path)


# %%
# =========================
# 15. OPTIONAL: PRINT PAPER-FRIENDLY TABLE
# =========================

paper_table = summary_df.copy()
for col in [
    "accuracy",
    "balanced_accuracy",
    "sensitivity_recall_seizure",
    "specificity_recall_nonseizure",
    "precision_seizure",
]:
    paper_table[col] = (paper_table[col] * 100).round(2)

paper_table["f1_seizure"] = paper_table["f1_seizure"].round(4)

paper_table = paper_table[
    [
        "patient",
        "accuracy",
        "balanced_accuracy",
        "sensitivity_recall_seizure",
        "specificity_recall_nonseizure",
        "precision_seizure",
        "f1_seizure",
        "tn",
        "fp",
        "fn",
        "tp",
    ]
]

display(paper_table)


W1012 13:20:07.571000 23680 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


GPU: NVIDIA GeForce RTX 5080
Precision: torch.bfloat16
Run folder:
C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\Raw\All training together RAw\MedGemma4B_IA3_Pooled_RAW\run_20261012_132008_593189

pooled RAW training: 2875 rows
CSV: C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\Raw\All training together RAw\training_metadata.csv
Image column: image_path
Label column: label
Class counts: {0: 2300, 1: 575}


Validate images: pooled RAW training:   0%|          | 0/2875 [00:00<?, ?it/s]


Train split: 2300 {0: 1840, 1: 460}
Val split: 575 {0: 460, 1: 115}


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

... (76 outputs left out)


Patient 24 RAW IA3
Accuracy:     89.80%
Balanced Acc: 81.25%
Sensitivity:  67.00%
Specificity:  95.50%
Precision:    78.82%
F1:           0.7243
CM [[TN, FP], [FN, TP]] = [[382, 18], [33, 67]]

Final patient-wise summary:


,patient,csv,predictions_csv,accuracy,balanced_accuracy,sensitivity_recall_seizure,specificity_recall_nonseizure,precision_seizure,f1_seizure,tn,fp,fn,tp
0,08,C:\Users\ga_sd360\OneDrive - The University of...,C:\Users\ga_sd360\OneDrive - The University of...,0.933,0.920625,0.900,0.94125,0.792952,0.843091,753,47,20,180
1,11,C:\Users\ga_sd360\OneDrive - The University of...,C:\Users\ga_sd360\OneDrive - The University of...,0.964,0.923125,0.855,0.99125,0.960674,0.904762,793,7,29,171
2,18,C:\Users\ga_sd360\OneDrive - The University of...,C:\Users\ga_sd360\OneDrive - The University of...,0.936,0.915000,0.880,0.95000,0.814815,0.846154,285,15,9,66
3,24,C:\Users\ga_sd360\OneDrive - The University of...,C:\Users\ga_sd360\OneDrive - The University of...,0.898,0.812500,0.670,0.95500,0.788235,0.724324,382,18,33,67


Saved summary to:
C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\Raw\All training together RAw\MedGemma4B_IA3_Pooled_RAW\run_20261012_132008_593189\patient_evaluations\summary_metrics.csv


,patient,accuracy,balanced_accuracy,sensitivity_recall_seizure,specificity_recall_nonseizure,precision_seizure,f1_seizure,tn,fp,fn,tp
0,08,93.3,92.06,90.0,94.12,79.30,0.8431,753,47,20,180
1,11,96.4,92.31,85.5,99.12,96.07,0.9048,793,7,29,171
2,18,93.6,91.50,88.0,95.00,81.48,0.8462,285,15,9,66
3,24,89.8,81.25,67.0,95.50,78.82,0.7243,382,18,33,67
